In [0]:
%run ./99_utils

In [0]:
setup_adls()

✅ ADLS Gen2 connected: clarityadls


✅ Clarity AML utilities loaded


In [0]:
def read_silver_transactions():
    return spark.read.parquet(silver("transactions"))

def read_bronze_sanctions_ofac():
    return spark.read.csv(
        bronze("reference/sanctions/ofac_sdn.csv"),
        header=False,
        inferSchema=False
    )

def read_bronze_kvk():
    return spark.read.csv(
        bronze("reference/kvk/kvk_companies.csv"),
        header=True,
        inferSchema=True
    )

# ── Write helpers ──────────────────────────────────────────────
def write_silver(df, path, partition_cols=["value_date_year", "value_date_month", "value_date_day"]):
    df.write \
      .mode("overwrite") \
      .partitionBy(*partition_cols) \
      .parquet(silver(path))
    print(f"✅ Written to Silver: {path}")

def write_gold(df, path, partition_cols=None):
    writer = df.write.mode("overwrite")
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.parquet(gold(path))
    print(f"✅ Written to Gold: {path}")

print("✅ Clarity AML utilities loaded")

def bronze(path=""):
    return f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/{path}"

def silver(path=""):
    return f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/{path}"

def gold(path=""):
    return f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/{path}"

✅ Clarity AML utilities loaded


In [0]:
%pip install thefuzz python-Levenshtein

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 20.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.1/153.1 kB 14.6 MB/s eta 0:00:00
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, StringType, BooleanType
from pyspark.sql.functions import udf, col, broadcast

process_date = datetime.now()
YEAR  = process_date.year
MONTH = process_date.month
DAY   = process_date.day

# ── BACKFILL MODE: load all dates, not just today ──────────────
# Switch this flag to False after first full run to return
# to daily incremental processing
BACKFILL_MODE = True

if BACKFILL_MODE:
    print("⚠️  BACKFILL MODE — loading ALL historical partitions")
    transactions = spark.read.parquet(silver("transactions"))
else:
    transactions = spark.read.parquet(
        silver("transactions")
    ).filter(
        (col("value_date_year")  == YEAR)  &
        (col("value_date_month") == MONTH) &
        (col("value_date_day")   == DAY)
    )

print(f"✅ Loaded: {transactions.count():,} records")
transactions.printSchema()

⚠️  BACKFILL MODE — loading ALL historical partitions
✅ Loaded: 1,575 records
root
 |-- transaction_id: string (nullable = true)
 |-- sender_iban: string (nullable = true)
 |-- sender_name: string (nullable = true)
 |-- sender_bic: string (nullable = true)
 |-- receiver_iban: string (nullable = true)
 |-- receiver_name: string (nullable = true)
 |-- receiver_bic: string (nullable = true)
 |-- amount_eur: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- purpose_code: string (nullable = true)
 |-- purpose_desc: string (nullable = true)
 |-- value_date: string (nullable = true)
 |-- booking_date: string (nullable = true)
 |-- ingestion_ts: long (nullable = true)
 |-- source_system: string (nullable = true)
 |-- message_type: string (nullable = true)
 |-- aml_pattern: string (nullable = true)
 |-- sender_name_clean: string (nullable = true)
 |-- receiver_name_clean: string (nullable = true)
 |-- sender_bic_clean: string (nullable = true)
 |-- receiver_bic_clean: string

In [0]:
# ── Counterparty frequency from full Silver history ────────────
# Read ALL historical Silver transactions (not just today)
# This tells us: has this sender→receiver pair transacted before?
print("📊 Computing counterparty frequency from Silver history...")

df_all_history = spark.read.parquet(silver("transactions")) \
    .select("sender_iban", "receiver_iban")

df_pair_counts = df_all_history.groupBy("sender_iban", "receiver_iban") \
    .agg(F.count("*").alias("historical_pair_count"))

# Join pair counts onto today's transactions
transactions = transactions.join(
    df_pair_counts,
    on=["sender_iban", "receiver_iban"],
    how="left"
).withColumn(
    "historical_pair_count",
    F.coalesce(F.col("historical_pair_count"), F.lit(0))
).withColumn(
    "counterparty_frequency",
    F.when(F.col("historical_pair_count") == 0,  "first_time")
     .when(F.col("historical_pair_count") < 5,   "occasional")
     .otherwise("established")
)

print(f"✅ Counterparty frequency computed")
transactions.groupBy("counterparty_frequency").count().show()

📊 Computing counterparty frequency from Silver history...
✅ Counterparty frequency computed
+----------------------+-----+
|counterparty_frequency|count|
+----------------------+-----+
|            occasional| 1376|
|           established|  199|
+----------------------+-----+



In [0]:
# ── Derive timing and purpose signals ─────────────────────────
# transaction_hour and day_of_week come from ingestion_ts
# purpose_name_mismatch is derived from purpose_code + sender_name
# No fabricated fields — everything comes from real SEPA data

transactions = transactions.withColumn(
    "transaction_hour",
    F.hour(F.to_timestamp(F.col("ingestion_ts") / 1000))
).withColumn(
    "day_of_week",
    F.dayofweek(F.to_timestamp(F.col("ingestion_ts") / 1000))
    # Spark: 1=Sunday, 2=Monday ... 7=Saturday
).withColumn(
    "purpose_name_mismatch",
    F.when(
        # Large "cash management" transfers are suspicious
        (F.col("purpose_code") == "CASH") & (F.col("amount_eur") > 5000),
        True
    ).when(
        # Dividend from a logistics/transport company is unusual
        (F.col("purpose_code") == "DIVI") &
        F.lower(F.col("sender_name")).rlike("transport|logistics|retail|construction"),
        True
    ).when(
        # Salary payment from something that doesn't look like a registered employer
        (F.col("purpose_code") == "SALA") &
        (~F.lower(F.col("sender_name")).rlike("bv|nv|holding|group|nv|cooperatie")),
        True
    ).when(
        # Intra-company transfer to a completely different named entity
        (F.col("purpose_code") == "INTC") &
        (F.col("sender_name_clean") != F.col("receiver_name_clean")),
        True
    ).otherwise(False)
)

print("✅ Timing and purpose signals derived")
transactions.select(
    "transaction_hour", "day_of_week", "purpose_name_mismatch", "purpose_code"
).show(5)

✅ Timing and purpose signals derived
+----------------+-----------+---------------------+------------+
|transaction_hour|day_of_week|purpose_name_mismatch|purpose_code|
+----------------+-----------+---------------------+------------+
|               8|          4|                false|        CASH|
|               8|          4|                false|        TRAD|
|              12|          4|                false|        CASH|
|              12|          4|                false|        TAXS|
|              12|          4|                false|        TRAD|
+----------------+-----------+---------------------+------------+
only showing top 5 rows



In [0]:
print("📂 Loading OFAC sanctions list...")

sanctions_raw = spark.read.csv(
    bronze("reference/sanctions/ofac_sdn.csv"),
    header=False,
    inferSchema=False
)

# Column_1 = entry ID, Column_2 = entity name, Column_4 = program
sanctions = sanctions_raw.select(
    col("_c0").alias("entry_id"),
    col("_c1").alias("entity_name"),
    col("_c3").alias("sanctions_program")
).filter(
    (col("entity_name").isNotNull()) &
    (col("entity_name") != "-0-") &
    (F.length(col("entity_name")) > 3)
).withColumn(
    "entity_name_clean",
    F.upper(F.trim(
        F.regexp_replace(
            F.regexp_replace(col("entity_name"), r'[^a-zA-Z0-9 ]', ''),
            r'\s+', ' '
        )
    ))
)

sanctions_count = sanctions.count()
print(f"✅ Loaded: {sanctions_count:,} sanctioned entities")
sanctions.show(5, truncate=False)

📂 Loading OFAC sanctions list...
✅ Loaded: 19,043 sanctioned entities
+--------+-------------------------+-----------------+----------------------+
|entry_id|entity_name              |sanctions_program|entity_name_clean     |
+--------+-------------------------+-----------------+----------------------+
|36      |AEROCARIBBEAN AIRLINES   |CUBA             |AEROCARIBBEAN AIRLINES|
|173     |ANGLO-CARIBBEAN CO., LTD.|CUBA             |ANGLOCARIBBEAN CO LTD |
|306     |BANCO NACIONAL DE CUBA   |CUBA             |BANCO NACIONAL DE CUBA|
|424     |BOUTIQUE LA MAISON       |CUBA             |BOUTIQUE LA MAISON    |
|475     |CASA DE CUBA             |CUBA             |CASA DE CUBA          |
+--------+-------------------------+-----------------+----------------------+
only showing top 5 rows



In [0]:
print("📂 Loading KvK company registry...")

kvk = spark.read.csv(
    bronze("reference/kvk/kvk_companies.csv"),
    header=True,
    inferSchema=True
).withColumn(
    "company_name_clean",
    F.upper(F.trim(
        F.regexp_replace(
            F.regexp_replace(col("company_name"), r'[^a-zA-Z0-9 ]', ''),
            r'\s+', ' '
        )
    ))
)

kvk_count = kvk.count()
print(f"✅ Loaded: {kvk_count:,} KvK companies")
kvk.show(5, truncate=False)

📂 Loading KvK company registry...
✅ Loaded: 50,000 KvK companies
+----------+-------------------------------------------+----------+-----------+----------------------+---------+--------+--------------+------------+-----------------+---------+------------------+--------------+-------------------------------------------+
|kvk_number|company_name                               |legal_form|sector_code|sector_description    |city     |postcode|street        |house_number|registration_date|status   |annual_revenue_eur|employee_count|company_name_clean                         |
+----------+-------------------------------------------+----------+-----------+----------------------+---------+--------+--------------+------------+-----------------+---------+------------------+--------------+-------------------------------------------+
|53269655  |Bosch Restaurants BV                       |BV        |5610       |Restaurants           |Almere   |8120 ZA |Jasonhof      |325         |2023-12-25       |

In [0]:
# ── Cash intensity anomaly using KvK capacity data ─────────────
# Revenue capacity derived from employee_count × sector rate.
# Uses your existing kvk DataFrame — no hardcoded benchmarks.
# A 2-person café cannot physically generate €8,000/day in cash.

from pyspark.sql.types import StructType, StructField, StringType, IntegerType

SECTOR_REVENUE_PER_EMPLOYEE = {
    "5610": 150,    # Restaurants
    "4771": 200,    # Retail clothing
    "4941": 400,    # Road freight
    "6201": 600,    # Software development
    "6419": 800,    # Financial services
    "6820": 1200,   # Real estate
    "7022": 700,    # Business consultancy
    "8299": 300,    # Business support
    "4649": 500,    # Wholesale household goods
    "6110": 900,    # Telecom
}

df_sector_rates = spark.createDataFrame(
    [(k, v) for k, v in SECTOR_REVENUE_PER_EMPLOYEE.items()],
    StructType([
        StructField("sector_code",     StringType(), False),
        StructField("revenue_per_emp", IntegerType(), False),
    ])
)

# Join sector rates onto KvK to get per-company capacity
kvk_with_capacity = kvk.join(
    df_sector_rates, "sector_code", "left"
).withColumn(
    "max_plausible_daily_eur",
    F.when(
        F.col("revenue_per_emp").isNotNull() &
        F.col("employee_count").isNotNull(),
        # × 2 = generous upper bound, not average
        F.col("employee_count") * F.col("revenue_per_emp") * 2
    ).otherwise(None)
).select(
    "company_name_clean",
    "employee_count",
    "max_plausible_daily_eur"
)

# Join capacity onto transactions via sender_name_clean
# sender_name_clean already exists from your ADF enrichment
transactions = transactions.join(
    kvk_with_capacity.select(
        F.col("company_name_clean").alias("sender_name_clean"),
        "max_plausible_daily_eur",
        "employee_count"
    ),
    "sender_name_clean",
    "left"
).withColumn(
    "cash_intensity_anomaly",
    F.when(
        (F.col("purpose_code") == "CASH") &
        (F.col("max_plausible_daily_eur").isNotNull()) &
        (F.col("amount_eur") > F.col("max_plausible_daily_eur")),
        True
    ).otherwise(False)
)

anomaly_count = transactions.filter(
    F.col("cash_intensity_anomaly")
).count()
print(f"✅ Cash intensity anomalies detected: {anomaly_count:,}")
transactions.filter(F.col("cash_intensity_anomaly")) \
    .select(
        "sender_name", "amount_eur",
        "max_plausible_daily_eur", "employee_count"
    ).show(5, truncate=False)

✅ Cash intensity anomalies detected: 0
+-----------+----------+-----------------------+--------------+
|sender_name|amount_eur|max_plausible_daily_eur|employee_count|
+-----------+----------+-----------------------+--------------+
+-----------+----------+-----------------------+--------------+



In [0]:
from thefuzz import fuzz
import re

def normalize_name(name: str) -> str:
    if not name:
        return ""
    name = re.sub(r'[^a-zA-Z0-9 ]', '', name)
    name = re.sub(r'\s+', ' ', name)
    return name.upper().strip()

def fuzzy_match_score(name1: str, name2: str) -> float:
    """
    Three algorithm ensemble:
    1. Token sort — handles reordered words
    2. Partial ratio — handles substrings
    3. Full ratio — handles typos
    """
    if not name1 or not name2:
        return 0.0
    n1 = normalize_name(name1)
    n2 = normalize_name(name2)

    token_sort = fuzz.token_sort_ratio(n1, n2) / 100.0
    partial    = fuzz.partial_ratio(n1, n2) / 100.0
    full       = fuzz.ratio(n1, n2) / 100.0

    return round((token_sort * 0.4) + (partial * 0.4) + (full * 0.2), 4)

print("✅ Fuzzy matching functions defined")
print("\nTest examples:")
print(f"  'BANCO NACIONAL CUBA' vs 'BANCO NACIONAL DE CUBA': "
      f"{fuzzy_match_score('BANCO NACIONAL CUBA', 'BANCO NACIONAL DE CUBA'):.2f}")
print(f"  'TIMCHENKO TRADING BV' vs 'TIMCHENKO': "
      f"{fuzzy_match_score('TIMCHENKO TRADING BV', 'TIMCHENKO'):.2f}")
print(f"  'VAN DER BERG BV' vs 'APPLE INC': "
      f"{fuzzy_match_score('VAN DER BERG BV', 'APPLE INC'):.2f}")

✅ Fuzzy matching functions defined

Test examples:
  'BANCO NACIONAL CUBA' vs 'BANCO NACIONAL DE CUBA': 0.91
  'TIMCHENKO TRADING BV' vs 'TIMCHENKO': 0.77
  'VAN DER BERG BV' vs 'APPLE INC': 0.29


In [0]:
# Collect sanctions names to driver
# We broadcast these so every Spark worker has a local copy
# instead of hitting the network for every comparison

sanctions_names_list = [
    row["entity_name_clean"]
    for row in sanctions.select("entity_name_clean").collect()
]

# Broadcast to all workers
sanctions_broadcast = spark.sparkContext.broadcast(sanctions_names_list)

print(f"✅ Broadcasted {len(sanctions_names_list):,} sanctions names to all workers")

✅ Broadcasted 19,043 sanctions names to all workers


In [0]:
MATCH_THRESHOLD = 0.85

def get_best_sanctions_score(name: str) -> float:
    """Find highest fuzzy match score against all sanctions."""
    if not name:
        return 0.0
    
    sanctions_list = sanctions_broadcast.value
    best_score = 0.0
    
    for s_name in sanctions_list:
        score = fuzzy_match_score(name, s_name)
        if score > best_score:
            best_score = score
        if best_score >= 0.99:
            break
    
    return float(best_score)

def get_best_sanctions_name(name: str) -> str:
    """Return the best matching sanctions entity name."""
    if not name:
        return ""
    
    sanctions_list = sanctions_broadcast.value
    best_score = 0.0
    best_name  = ""
    
    for s_name in sanctions_list:
        score = fuzzy_match_score(name, s_name)
        if score > best_score:
            best_score = score
            best_name  = s_name
    
    return best_name if best_score >= MATCH_THRESHOLD else ""

# Register as Spark UDFs
sanctions_score_udf = udf(get_best_sanctions_score, FloatType())
sanctions_name_udf  = udf(get_best_sanctions_name,  StringType())

print(f"✅ UDFs registered")
print(f"   Match threshold: {MATCH_THRESHOLD}")

✅ UDFs registered
   Match threshold: 0.85


In [0]:
print("🔍 Step 1: Exact matching...")



# ── Exact sanctions match ──────────────────────────────────────
exact_sanctions = transactions.join(
    broadcast(sanctions.select(
        "entity_name_clean",
        "sanctions_program"
    )),
    transactions["sender_name_clean"] == sanctions["entity_name_clean"],
    "left"
).withColumn(
    "sanctions_exact_hit",
    col("sanctions_program").isNotNull()
)

exact_hits = exact_sanctions.filter(col("sanctions_exact_hit")==True).count()
print(f"   Exact sanctions hits: {exact_hits:,}")

# ── Exact KvK match ────────────────────────────────────────────
with_kvk = exact_sanctions.join(
    broadcast(kvk.select(
        "company_name_clean",
        "kvk_number",
        "status",
        "sector_description",
        "registration_date",
        "city"
    )),
    exact_sanctions["sender_name_clean"] == kvk["company_name_clean"],
    "left"
).withColumn(
    "kvk_registered",
    col("kvk_number").isNotNull()
).withColumn(
    "kvk_status",
    F.when(col("status").isNull(), "UNKNOWN")
     .otherwise(col("status"))
)

# Split into matched and unmatched for fuzzy processing
unmatched = with_kvk.filter(col("sanctions_exact_hit") == False)
matched   = with_kvk.filter( col("sanctions_exact_hit") == True)

unmatched.show()

print(f"   Unmatched (needs fuzzy): {unmatched.count():,}")
print(f"   Already matched (exact): {matched.count():,}")

kvk_hits = with_kvk.filter(col("kvk_registered")).count()
print(f"   Exact KvK hits:       {kvk_hits:,}")

🔍 Step 1: Exact matching...
   Exact sanctions hits: 125
+--------------------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+------------+----------+--------+------------+----------------+----------+------------+-------------+-------------+------------+--------------------+--------------------+----------------+------------------+------------------+-------------------+--------------------+--------------------+----------+-------------+---------------+----------------+--------------+---------------------+----------------------+----------------+-----------+---------------------+-----------------------+--------------+----------------------+-----------------+-----------------+-------------------+--------------------+----------+------+--------------------+-----------------+---------+--------------+----------+
|   sender_name_clean|         sender_iban|       receiver_iban|      transaction_id|         sender_name|sender_bic

In [0]:
print("\n🔍 Step 2: Optimised fuzzy matching...")

from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType, StringType

# ── Step 2a — Blocking key UDF ─────────────────────────────────
def get_blocking_key(name: str) -> str:
    if not name:
        return ""
    words = name.split()
    for word in words:
        if len(word) >= 4:
            return word
    return words[0] if words else ""

blocking_key_udf = udf(get_blocking_key, StringType())


# ── Step 2b — Add blocking keys ────────────────────────────────
unmatched_keyed = unmatched.withColumn(
    "blocking_key",
    blocking_key_udf(col("sender_name_clean"))
).alias("txn")

sanctions_keyed = sanctions.withColumn(
    "blocking_key",
    blocking_key_udf(col("entity_name_clean"))
).select(
    "blocking_key",
    col("entity_name_clean").alias("sanctions_name_clean"),
    col("sanctions_program").alias("sanction_program")
).alias("san")

# ── Step 2c — Join on blocking key ─────────────────────────────
# Now no ambiguous columns — txn.entity_name_clean doesn't exist
# and sanctions uses sanctions_name_clean
candidates = unmatched_keyed.join(
    broadcast(sanctions_keyed),
    col("txn.blocking_key") == col("san.blocking_key"),
    "left"
)

candidate_count = candidates.count()
print(f"   Candidate pairs: {candidate_count:,}")

# ── Step 2d — Score each candidate pair ────────────────────────
@udf(FloatType())
def score_pair(name1: str, name2: str) -> float:
    if not name1 or not name2:
        return 0.0
    from thefuzz import fuzz
    import re
    def norm(n):
        n = re.sub(r'[^a-zA-Z0-9 ]', '', n)
        n = re.sub(r'\s+', ' ', n)
        return n.upper().strip()
    n1, n2 = norm(name1), norm(name2)
    token_sort = fuzz.token_sort_ratio(n1, n2) / 100.0
    partial    = fuzz.partial_ratio(n1, n2) / 100.0
    full       = fuzz.ratio(n1, n2) / 100.0
    return round((token_sort * 0.4) + (partial * 0.4) + (full * 0.2), 4)

scored = candidates.withColumn(
    "fuzzy_score",
    score_pair(
        col("txn.sender_name_clean"),
        col("san.sanctions_name_clean")
    )
)

# ── Step 2e — Keep only matches above threshold ─────────────────
fuzzy_matches = scored.filter(
    col("fuzzy_score") >= MATCH_THRESHOLD
).groupBy(
    col("txn.transaction_id").alias("transaction_id")
).agg(
    F.max("fuzzy_score").alias("fuzzy_sanctions_score"),
    F.first("san.sanctions_name_clean").alias("fuzzy_matched_name"),
    F.first("san.sanction_program").alias("fuzzy_sanctions_program")
)

fuzzy_hit_count = fuzzy_matches.count()
print(f"   Fuzzy matches found: {fuzzy_hit_count:,}")

# ── Step 2f — Join matches back to unmatched transactions ───────
fuzzy_results = unmatched.join(
    fuzzy_matches,
    "transaction_id",
    "left"
).withColumn(
    "fuzzy_sanctions_hit",
    col("fuzzy_sanctions_score").isNotNull()
).withColumn(
    "fuzzy_sanctions_score",
    F.when(col("fuzzy_sanctions_score").isNull(), 0.0)
     .otherwise(col("fuzzy_sanctions_score"))
).withColumn(
    "fuzzy_matched_name",
    F.when(col("fuzzy_matched_name").isNull(), "")
     .otherwise(col("fuzzy_matched_name"))
)

fuzzy_hits = fuzzy_results.filter(col("fuzzy_sanctions_hit")).count()
print(f"✅ Fuzzy sanctions hits: {fuzzy_hits:,}")


🔍 Step 2: Optimised fuzzy matching...
   Candidate pairs: 2,978
   Fuzzy matches found: 0
✅ Fuzzy sanctions hits: 0


In [0]:
# Show the 2 sanctions hits in detail
print("🚨 Sanctions hits found:")

fuzzy_results.filter(col("fuzzy_sanctions_hit")) \
    .select(
        "transaction_id",
        "sender_name",
        "sender_name_clean",
        "fuzzy_matched_name",
        "fuzzy_sanctions_score",
        "amount_eur"
    ).show(10, truncate=False)

🚨 Sanctions hits found:
+--------------+-----------+-----------------+------------------+---------------------+----------+
|transaction_id|sender_name|sender_name_clean|fuzzy_matched_name|fuzzy_sanctions_score|amount_eur|
+--------------+-----------+-----------------+------------------+---------------------+----------+
+--------------+-----------+-----------------+------------------+---------------------+----------+



In [0]:
from pyspark.sql.functions import lit, current_timestamp, when

print("\n🔗 Combining exact and fuzzy results...")

# Add fuzzy columns to exact matches
exact_with_fuzzy = matched \
    .withColumn("fuzzy_sanctions_score", lit(1.0).cast(FloatType())) \
    .withColumn("fuzzy_sanctions_hit",   lit(True)) \
    .withColumn("fuzzy_matched_name",    col("sender_name_clean"))

# Align columns for union
fuzzy_results_aligned = fuzzy_results.select(
    *exact_with_fuzzy.columns
)

# Union both
final = exact_with_fuzzy.unionByName(fuzzy_results_aligned)

# Add final consolidated flags
final = final \
    .withColumn(
        "sanctions_hit",
        col("sanctions_exact_hit") | col("fuzzy_sanctions_hit")
    ) \
    .withColumn(
        "match_method",
        when(col("sanctions_exact_hit"),  "EXACT")
        .when(col("fuzzy_sanctions_hit"), "FUZZY")
        .otherwise("NO_MATCH")
    ) \
    .withColumn(
        "fuzzy_match_completed_at",
        current_timestamp()
    ) \
    .withColumn(
        "processing_status",
        lit("FUZZY_MATCH_COMPLETE")
    )

# ── Risk scoring ───────────────────────────────────────────────
# Added after building `final`, before writing to Silver
# All signals come from real SEPA data — nothing fabricated

# ── Risk scoring ───────────────────────────────────────────────
# Add this block after final is built, before the .write call
# All columns referenced here were added in Cells A, B, C above

final = final.withColumn(
    "amount_risk",
    F.when(F.col("amount_eur") > 100000, 3)
     .when(F.col("amount_eur") > 50000,  2)
     .when(F.col("amount_eur") > 10000,  1)
     .otherwise(0)
).withColumn(
    "timing_risk",
    F.when(F.col("transaction_hour").between(0, 5), 2)
     .when(F.col("day_of_week").isin([1, 7]), 1)
     .otherwise(0)
).withColumn(
    "relationship_risk",
    F.when(F.col("counterparty_frequency") == "first_time",  2)
     .when(F.col("counterparty_frequency") == "occasional",  1)
     .otherwise(0)
).withColumn(
    "purpose_risk",
    F.when(F.col("purpose_name_mismatch") == True, 2)
     .otherwise(0)
).withColumn(
    "sanctions_risk",
    F.when(F.col("sanctions_hit") == True, 5)
     .otherwise(0)
).withColumn(
    "sector_anomaly_risk",
    F.when(F.col("cash_intensity_anomaly") == True, 3)
     .otherwise(0)
).withColumn(
    "composite_risk_score",
    F.col("amount_risk") +
    F.col("timing_risk") +
    F.col("relationship_risk") +
    F.col("purpose_risk") +
    F.col("sanctions_risk") +
    F.col("sector_anomaly_risk")
).withColumn(
    "risk_tier",
    F.when(F.col("composite_risk_score") >= 7, "HIGH")
     .when(F.col("composite_risk_score") >= 4, "MEDIUM")
     .otherwise("LOW")
)

print("✅ Risk scores computed")
final.groupBy("risk_tier").count().orderBy("risk_tier").show()



# ── Write to Silver enriched ───────────────────────────────────
output_path = silver("transactions_enriched")

spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
final.write \
    .mode("overwrite") \
    .partitionBy("value_date_year", "value_date_month", "value_date_day") \
    .parquet(output_path)

print(f"✅ Written to Silver enriched: {output_path}")
print(f"   Total records written: {final.count():,}")


🔗 Combining exact and fuzzy results...
✅ Risk scores computed
+---------+-----+
|risk_tier|count|
+---------+-----+
|     HIGH|   67|
|      LOW| 2302|
|   MEDIUM|  319|
+---------+-----+

✅ Written to Silver enriched: abfss://silver@clarityadls.dfs.core.windows.net/transactions_enriched
   Total records written: 2,688
